# 0.&nbsp;Hướng dẫn sử dụng

Notebook này minh họa kết quả thực nghiệm cho bài tập lớn số 1, bao gồm:
1. Chuyển đổi không gian màu và thao tác kênh ảnh.
2. Áp dụng các bộ lọc làm trơn (Low-pass) và tách biên (High-pass).

Hướng dẫn chạy: Nhấn nút `Run all` để chạy toàn bộ dự án.

# 1.&nbsp;Cài đặt thư viện và chuẩn bị dữ liệu

In [ ]:
import os
import sys

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

np.random.seed(42)

In [ ]:
GITHUB_USER = "broistg"
REPO_NAME = "CV-Project-2_Nhom-18"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
ROOT_DIR = "/content"
INPUT_DIR = "data/inputs"
OUTPUT_DIR = "data/outputs"

repo_absolute_path = os.path.join(ROOT_DIR, REPO_NAME)

# Clone Repository nếu chưa có
if not os.path.exists(repo_absolute_path):
    print(f"Cloning {REPO_NAME} from {REPO_URL}...")
    !git clone $REPO_URL
    print(f"Repository {REPO_NAME} cloned successfully into {repo_absolute_path}.")
else:
    print(f"Repository {REPO_NAME} already exists at {repo_absolute_path}.")

# Thiết lập thư mục làm việc
if os.path.abspath(os.getcwd()) != repo_absolute_path:
    os.chdir(repo_absolute_path)
    print(f"Changed current working directory to: {os.getcwd()}")
else:
    print(f"Current working directory is already: {os.getcwd()}")

# Thêm vào sys.path
if repo_absolute_path not in sys.path:
    sys.path.insert(0, repo_absolute_path)
    print(f"Added {repo_absolute_path} to sys.path.")
else:
    print(f"{repo_absolute_path} is already in sys.path.")

# Import module tự viết
try:
    from src import gradient_editing, geometry_transform, homography, utils
    print("Custom modules (gradient_editing, geometry_transform, homography, utils) imported successfully from src.")
except ImportError as e:
    print(f"❌ Lỗi Import: {e}. Please ensure 'src' directory exists in {repo_absolute_path} and contains an __init__.py file for package recognition.")
    print("Current sys.path:", sys.path)

# Projective (homography) transformation.

## Load ảnh

In [ ]:
ad_screen_name = "geometry/bg1.jpg"
ad_screen_path = os.path.join(INPUT_DIR, ad_screen_name)
ad_screen_rgb = utils.read_image(ad_screen_path)

building_name = "geometry/bg2.jpg"
building_path = os.path.join(INPUT_DIR, building_name)
building_rgb = utils.read_image(building_path)

karpathy_name = "geometry/karpathy.jpeg"
karpathy_path = os.path.join(INPUT_DIR, karpathy_name)
karpathy_rgb = utils.read_image(karpathy_path)

utils.show_comparison(ad_screen_rgb, building_rgb, "Ad Screen", "Building")

## Hiện thực

In [ ]:
ad_screen_name = "geometry/bg1.jpg"
ad_screen_path = os.path.join(INPUT_DIR, ad_screen_name)
ad_screen = utils.read_image(ad_screen_path)
ad_screen = cv2.resize(ad_screen, (ad_screen.shape[1]//10, ad_screen.shape[0]//10), interpolation=cv2.INTER_AREA)

ad_screen_src_pts = np.float32([[105, 60], [210, 100], [205, 270], [85, 265]])
ad_screen_dst_pts = np.float32([[105, 100], [210, 100], [210, 270], [105, 270]])

ad_screen_src = ad_screen.copy()

for pt in ad_screen_src_pts:
    cv2.circle(ad_screen_src, np.int32(tuple(pt)), 3, (0, 255, 0), -1)

H, mask = homography.get_homography(ad_screen_src_pts, ad_screen_dst_pts)
ad_screen_dst = homography.apply_transform(ad_screen, H)

for pt in ad_screen_dst_pts:
    cv2.circle(ad_screen_dst, np.int32(tuple(pt)), 3, (0, 255, 0), -1)

utils.show_comparison(ad_screen_src, ad_screen_dst, "Original", "Warped (Auto-size)")

In [ ]:
building_name = "geometry/bg2.jpg"
building_path = os.path.join(INPUT_DIR, building_name)
building = utils.read_image(building_path)
building = cv2.resize(building, (building.shape[1]//10, building.shape[0]//10), interpolation=cv2.INTER_AREA)

building_src_pts = np.float32([[200, 0], [205, 380], [60, 370], [120, 80]])
building_dst_pts = np.float32([[200, 0], [210, 400], [0, 400], [10, 90]])
building_src = building.copy()

for pt in building_src_pts:
    cv2.circle(building_src, np.int32(tuple(pt)), 3, (0, 255, 0), -1)

H, mask = homography.get_homography(building_src_pts, building_dst_pts)
building_dst = homography.apply_transform(building, H)

for pt in building_dst_pts:
    cv2.circle(building_dst, np.int32(tuple(pt)), 3, (0, 255, 0), -1)

utils.show_comparison(building_src, building_dst, "Original", "Warped (Auto-size)")

# Thử nghiệm mở rộng: dán một hình chữ nhật hoặc hình vuông lên một mặt phẳng được chỉ định trước

## Thủ công:

In [ ]:
ad_screen_name = "geometry/bg1.jpg"
ad_screen_path = os.path.join(INPUT_DIR, ad_screen_name)
ad_screen = utils.read_image(ad_screen_path)
ad_screen = cv2.resize(ad_screen, (ad_screen.shape[1]//10, ad_screen.shape[0]//10), interpolation=cv2.INTER_AREA)

karpathy_name = "geometry/karpathy.jpeg"
karpathy_path = os.path.join(INPUT_DIR, karpathy_name)
karpathy = utils.read_image(karpathy_path)

h_karpathy, w_karpathy = karpathy.shape[:2]
karpathy_src_pts = np.float32([[0, 0], [w_karpathy, 0], [w_karpathy, h_karpathy], [0, h_karpathy]])
ad_screen_dst_pts = np.float32([[105, 60], [210, 100], [205, 270], [85, 265]])

H, _ = homography.get_homography(karpathy_src_pts, ad_screen_dst_pts)

result_manual = homography.overlay_image(ad_screen, karpathy, H)

utils.show_comparison(ad_screen, result_manual, "Original", "Warped (Auto-size)")

In [ ]:
building_name = "geometry/bg2.jpg"
building_path = os.path.join(INPUT_DIR, building_name)
building = utils.read_image(building_path)
building = cv2.resize(building, (building.shape[1]//10, building.shape[0]//10), interpolation=cv2.INTER_AREA)

karpathy_name = "geometry/karpathy.jpeg"
karpathy_path = os.path.join(INPUT_DIR, karpathy_name)
karpathy = utils.read_image(karpathy_path)

h_karpathy, w_karpathy = karpathy.shape[:2]
karpathy_src_pts = np.float32([[0, 0], [w_karpathy, 0], [w_karpathy, h_karpathy], [0, h_karpathy]])
building_dst_pts = np.float32([[200, 0], [205, 380], [60, 370], [120, 80]])

H, _ = homography.get_homography(karpathy_src_pts, building_dst_pts)

result_manual = homography.overlay_image(building, karpathy, H)

utils.show_comparison(building, result_manual, "Original", "Warped (Auto-size)")

## Tự động:

In [ ]:
ad_screen_name = "geometry/bg1.jpg"
ad_screen_path = os.path.join(INPUT_DIR, ad_screen_name)
ad_screen = utils.read_image(ad_screen_path)
ad_screen = cv2.resize(ad_screen, (ad_screen.shape[1]//10, ad_screen.shape[0]//10), interpolation=cv2.INTER_AREA)

template_name = "geometry/template_bg1.png"
template_path = os.path.join(INPUT_DIR, template_name)
template = utils.read_image(template_path)

karpathy_name = "geometry/karpathy.jpeg"
karpathy_path = os.path.join(INPUT_DIR, karpathy_name)
karpathy = utils.read_image(karpathy_path)

h_karpathy, w_karpathy = karpathy.shape[:2]
karpathy_src_pts = np.float32([[0, 0], [w_karpathy, 0], [w_karpathy, h_karpathy], [0, h_karpathy]])

found_corners, H = homography.automatic_find_dst_pts(template, ad_screen)

H, _ = homography.get_homography(karpathy_src_pts, found_corners)

result_auto = homography.overlay_image(ad_screen, karpathy, H)

utils.show_comparison(ad_screen, result_auto, "Original", "Warped (Auto-size)")